# Reading a 305 MB satellite image by moving 32 KB

**What this proves:** a Cloud-Optimized GeoTIFF (COG) can be read over plain HTTP without downloading it. The server answers a byte-range request with `206 Partial Content`, so a client fetches only the header and the tiles it needs.

**The scene:** Sentinel-2 true colour, tile 30SVF, 15 August 2025, 0 percent cloud, covering the Granada coast. Provenance in [`data/SOURCE.md`](data/SOURCE.md).

**Result:** 305.6 MB on the server, 32,768 bytes moved. The header read shows why: 1024 x 1024 internal tiles and overviews at levels 2, 4, 8 and 16.

**The map is not in this notebook.** It was made in QGIS, streaming the same COG with the Punta de la Mona MPA boundary on top: [`punta_de_la_mona_sentinel2_map.png`](punta_de_la_mona_sentinel2_map.png)

![Sentinel-2 map of Punta de la Mona MPA](punta_de_la_mona_sentinel2_map.png)


In [1]:
import requests

url = ("https://sentinel-cogs.s3.us-west-2.amazonaws.com/"
       "sentinel-s2-l2a-cogs/30/S/VF/2025/8/S2C_30SVF_20250815_0_L2A/TCI.tif")

head = requests.head(url)
print("full file size, MB:", round(int(head.headers["Content-Length"]) / 1e6, 1))
print("server accepts byte ranges:", head.headers.get("Accept-Ranges"))

r = requests.get(url, headers={"Range": "bytes=0-32767"})
print("status code:", r.status_code, "(206 = partial content)")
print("bytes actually downloaded:", len(r.content))

full file size, MB: 305.6
server accepts byte ranges: bytes
status code: 206 (206 = partial content)
bytes actually downloaded: 32768


In [2]:
!pip install rasterio -q

import rasterio

with rasterio.open(url) as src:
    print("size in pixels:", src.width, "x", src.height)
    print("CRS:", src.crs)
    print("bands:", src.count, " dtype:", src.dtypes[0])
    print("pixel size, metres:", src.res)
    print("internal tiling:", src.profile.get("blockxsize"), "x", src.profile.get("blockysize"))
    print("overview levels:", src.overviews(1))

size in pixels: 10980 x 10980
CRS: EPSG:32630
bands: 3  dtype: uint8
pixel size, metres: (10.0, 10.0)
internal tiling: 1024 x 1024
overview levels: [2, 4, 8, 16]
